# CS 5330 Mini-Project 3: Voice-Driven Visual Inspector

<a href="https://colab.research.google.com/github/LinoNU/cs5330-mp3-visa/blob/main/notebooks/mp3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this mini-project you'll build a small inspection tool. A user shows it an image of a food product, asks a question out loud ("Is this cashew damaged? Show me where."), and the tool answers in text and highlights the defect on the image.

The pipeline has four parts:

1. **Speech to text:** Whisper transcribes the spoken question.
2. **Intent:** the transcript is mapped to an action, such as *detect* (is it defective?), *locate* (where?), or *describe*.
3. **Detection and localization:** your vision method produces an anomaly score for the image and a heatmap showing where the defect is. You'll implement and compare two approaches: a **zero-shot** method using CLIP and text prompts, and a **few-shot** method using a memory bank of patch features from normal reference images.
4. **Interface:** a Gradio app that ties everything together.

### What this notebook gives you

- Data loading for the curated VisA dataset
- Metric functions, so everyone computes results the same way
- An evaluation harness that runs any method on a split and reports image-level AUROC, pixel-level AUROC, and IoU, overall and per defect type
- A simple classical baseline (difference from a mean reference image) so you can see the harness working end to end
- Whisper transcription and a working Gradio shell

### What you build

- The zero-shot CLIP method (Section 6)
- The few-shot memory-bank method (Section 7)
- Intent parsing and the answer logic for the app (Section 9)
- Your experiments, evaluation, and failure analysis (Section 8 and your report)

Cells marked **TODO** are yours to complete. Please don't change the metric functions in Section 3: keeping them fixed makes results comparable across the class.

**Runtime:** use a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

## 1. Setup

In [ ]:
# Gradio isn't preinstalled in Colab. Everything else used here is.
%pip install --quiet gradio

In [ ]:
### added after
import json
import requests
import gradio as gr
import librosa
import soundfile as sf
import warnings

In [ ]:
import random
import subprocess
import time
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score

SEED = 5330
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("No GPU found. Switch to a GPU runtime before running the model sections.")

Record the library versions you used in your report so that your results can be reproduced.

In [ ]:
import importlib.metadata as metadata

for package in ["torch", "transformers", "gradio", "numpy", "pandas", "scikit-learn"]:
    try:
        print(f"{package:>13}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:>13}: not installed")

## 2. Data

The dataset lives in the [cs5330-mp3-visa](https://github.com/LinoNU/cs5330-mp3-visa) repository. Read its README for the details: which subsets are core and which are extensions, how the splits work, and what each manifest column means.

The key rule: **tune on `dev`, report on `test`.** The `reference` split contains only normal images; your few-shot method draws its memory bank from it.

In [ ]:
REPO_URL = "https://github.com/LinoNU/cs5330-mp3-visa.git"
REPO_DIR = Path("cs5330-mp3-visa")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

manifest = pd.read_csv(REPO_DIR / "data" / "manifest.csv", keep_default_na=False)
manifest["defect_area_px"] = pd.to_numeric(manifest["defect_area_px"], errors="coerce")

CORE_SUBSETS = ["cashew", "macaroni1"]
print(f"{len(manifest)} images in {manifest['subset'].nunique()} subsets")
manifest.pivot_table(
    index="subset",
    columns=["split", "label"],
    values="image_path",
    aggfunc="count"
).fillna(0).astype(int)

In [ ]:
@dataclass(frozen=True)
class Sample:
    # One image from the manifest, loaded into memory.

    image: Image.Image          # RGB
    mask: np.ndarray            # bool, shape (H, W); all False for normal images
    label: int                  # 1 = anomaly, 0 = normal
    defect_type: str            # "" for normal images
    defect_area_px: float       # NaN for normal images
    image_path: str


def load_sample(row: pd.Series) -> Sample:
    # Load the image (and its mask, if any) for one manifest row.
    image = Image.open(REPO_DIR / row["image_path"]).convert("RGB")
    if row["mask_path"]:
        mask = np.asarray(Image.open(REPO_DIR / row["mask_path"])) == 255
    else:
        mask = np.zeros((image.height, image.width), dtype=bool)
    return Sample(
        image=image,
        mask=mask,
        label=int(row["label"] == "anomaly"),
        defect_type=row["defect_type"],
        defect_area_px=row["defect_area_px"],
        image_path=row["image_path"],
    )

"""
row = manifest[(manifest["subset"] == subset) &
               (manifest["split"] == split) ]
"""
def get_rows(subset: str, split: str) -> pd.DataFrame:
    # Return the manifest rows for one subset and split, in a fixed order.
    rows = manifest.query("subset == @subset and split == @split")
    if rows.empty:
        raise ValueError(f"No rows for subset={subset!r}, split={split!r}")
    return rows.sort_values("image_path").reset_index(drop=True)


def load_split(subset: str, split: str) -> list[Sample]:
    return [load_sample(row) for _, row in get_rows(subset, split).iterrows()]

In [ ]:
def overlay_heatmap(
    image: Image.Image, heatmap: np.ndarray, alpha: float = 0.5, cmap: str = "jet"
) -> Image.Image:
    # Blend a heatmap (any range, shape (H, W)) over an image.
    heatmap = resize_heatmap(heatmap, image.size)
    span = heatmap.max() - heatmap.min()
    normalized = (heatmap - heatmap.min()) / span if span > 0 else np.zeros_like(heatmap)
    coloured = (plt.get_cmap(cmap)(normalized)[..., :3] * 255).astype(np.uint8)
    return Image.blend(image.convert("RGB"), Image.fromarray(coloured), alpha)


def outline_mask(image: Image.Image, mask: np.ndarray, colour=(200, 16, 46)) -> Image.Image:
    # Draw the boundary of a boolean mask on an image.
    from scipy.ndimage import binary_erosion

    edge = mask & ~binary_erosion(mask, iterations=2)
    pixels = np.asarray(image.convert("RGB")).copy()
    pixels[edge] = colour
    return Image.fromarray(pixels)


def show_samples(samples: Sequence[Sample], n: int = 4) -> None:
    # Show a few images with their ground-truth defect outlined in red.
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, sample in zip(np.atleast_1d(axes), samples[:n]):
        ax.imshow(outline_mask(sample.image, sample.mask))
        title = sample.defect_type or "normal"
        ax.set_title(f"{title}\n{Path(sample.image_path).name}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Look at the data before modelling it.
dev_cashew = load_split("cashew", "dev")
show_samples([s for s in dev_cashew if s.label == 1], n=4)
show_samples([s for s in dev_cashew if s.label == 0], n=4)

## 3. Metrics (provided; please don't modify)

Every method you build is evaluated through the same interface:

- an **image score**: one float per image, higher means more likely defective
- a **heatmap**: a 2-D float array, higher means more likely defective at that location. It can be any size; it's resized to the image size before scoring.

The metrics are:

| Metric | Question it answers |
| --- | --- |
| Image AUROC | Does the image score rank defective images above normal ones? |
| Pixel AUROC | Does the heatmap rank defect pixels above normal pixels, across all images in the split? |
| Mean IoU | After thresholding the heatmap, how well does the highlighted region overlap the true defect? Computed on anomalous images only. |

AUROC doesn't need a threshold, but IoU does. Choose the IoU threshold on `dev` with `best_iou_threshold` and then apply the *same* threshold to `test`.

In [ ]:
def resize_heatmap(heatmap: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    # Resize a 2-D heatmap to size=(width, height) with bilinear interpolation.
    heatmap = np.asarray(heatmap, dtype=np.float32)
    if heatmap.ndim != 2:
        raise ValueError(f"Heatmap must be 2-D, got shape {heatmap.shape}")
    if (heatmap.shape[1], heatmap.shape[0]) == tuple(size):
        return heatmap
    resized = Image.fromarray(heatmap).resize(size, Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)


def image_auroc(labels: Sequence[int], scores: Sequence[float]) -> float:
    return float(roc_auc_score(labels, scores))


def pixel_auroc(masks: Sequence[np.ndarray], heatmaps: Sequence[np.ndarray]) -> float:
    # AUROC over every pixel of every image (heatmaps already at image size).
    y_true = np.concatenate([m.ravel() for m in masks])
    y_score = np.concatenate([h.ravel() for h in heatmaps])
    return float(roc_auc_score(y_true, y_score))


def mask_iou(predicted: np.ndarray, target: np.ndarray) -> float:
    # Intersection over union of two boolean masks.
    union = np.logical_or(predicted, target).sum()
    if union == 0:
        return float("nan")
    return float(np.logical_and(predicted, target).sum() / union)


def best_iou_threshold(
    masks: Sequence[np.ndarray], heatmaps: Sequence[np.ndarray], n_candidates: int = 50
) -> float:
    # Pick the heatmap threshold that maximizes mean IoU on anomalous images.
    # Call this on dev predictions only, then reuse the threshold on test.
    anomalous = [(m, h) for m, h in zip(masks, heatmaps) if m.any()]
    values = np.concatenate([h.ravel() for _, h in anomalous])
    candidates = np.quantile(values, np.linspace(0.50, 0.999, n_candidates))
    mean_ious = [
        np.nanmean([mask_iou(h >= t, m) for m, h in anomalous]) for t in candidates
    ]
    return float(candidates[int(np.argmax(mean_ious))])

## 4. Evaluation harness (provided)

A *method* is any object with a `predict(image)` method that returns `(score, heatmap)`. Methods that learn from normal images (like the few-shot one) also have a `fit(reference_images)` method.

`run_method` applies a method to a list of samples and times it. `evaluate` computes the metrics and returns a per-image table that you can use for failure analysis.

In [ ]:
@dataclass(frozen=True)
class Prediction:
    score: float
    heatmap: np.ndarray  # resized to the image size
    seconds: float       # inference time for this image


def run_method(method, samples: Sequence[Sample]) -> list[Prediction]:
    # Run method.predict on every sample.
    predictions = []
    for sample in samples:
        start = time.perf_counter()
        score, heatmap = method.predict(sample.image)
        elapsed = time.perf_counter() - start
        predictions.append(
            Prediction(float(score), resize_heatmap(heatmap, sample.image.size), elapsed)
        )
    return predictions


def evaluate(
    samples: Sequence[Sample],
    predictions: Sequence[Prediction],
    iou_threshold: float | None = None,
) -> tuple[dict[str, float], pd.DataFrame]:
    # Compute metrics. Pass iou_threshold (chosen on dev) to also report mean IoU.
    labels = [s.label for s in samples]
    masks = [s.mask for s in samples]
    heatmaps = [p.heatmap for p in predictions]

    metrics = {
        "image_auroc": image_auroc(labels, [p.score for p in predictions]),
        "pixel_auroc": pixel_auroc(masks, heatmaps),
        "seconds_per_image": float(np.mean([p.seconds for p in predictions])),
    }

    per_image = pd.DataFrame(
        {
            "image_path": [s.image_path for s in samples],
            "label": labels,
            "defect_type": [s.defect_type for s in samples],
            "defect_area_px": [s.defect_area_px for s in samples],
            "score": [p.score for p in predictions],
        }
    )
    if iou_threshold is not None:
        per_image["iou"] = [
            mask_iou(h >= iou_threshold, m) if m.any() else np.nan
            for m, h in zip(masks, heatmaps)
        ]
        metrics["mean_iou"] = float(per_image["iou"].mean())
        metrics["iou_threshold"] = float(iou_threshold)
    return metrics, per_image


def by_defect_type(per_image: pd.DataFrame, min_images: int = 5) -> pd.DataFrame:
    # Summarize anomalous images per defect type. An image with several defects
    # (e.g. "burnt,small holes") counts towards each of its types, and its IoU
    # is for the whole mask. Image AUROC compares each type with all normal
    # images in the split. Types with fewer than min_images images are flagged:
    # their AUROC rests on too few examples to mean much.
    normal_scores = per_image.loc[per_image["label"] == 0, "score"]
    anomalous = per_image[per_image["label"] == 1].copy()
    anomalous["defect_type"] = anomalous["defect_type"].str.split(",")
    anomalous = anomalous.explode("defect_type")
    anomalous["defect_type"] = anomalous["defect_type"].str.strip()
    rows = []
    for defect_type, group in anomalous.groupby("defect_type"):
        scores = pd.concat([group["score"], normal_scores])
        labels = [1] * len(group) + [0] * len(normal_scores)
        row = {
            "defect_type": defect_type,
            "images": len(group),
            "reliable": len(group) >= min_images,
            "median_area_px": group["defect_area_px"].median(),
            "image_auroc": image_auroc(labels, scores),
        }
        if "iou" in group:
            row["mean_iou"] = group["iou"].mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values("images", ascending=False, ignore_index=True)

## 5. A classical baseline (provided)

Before using deep features, it helps to know how far a very simple method gets. This baseline averages the *k* reference images and scores each pixel of a new image by how much its colour differs from that average, after a little smoothing.

It works best when objects are aligned and look alike. Think about how cashew and macaroni1 differ in that respect before you run it, and check whether the results match your prediction.

In [ ]:
class MeanImageBaseline:
    # Anomaly = smoothed per-pixel colour difference from the mean reference image.

    def __init__(self, sigma: float = 4.0) -> None:
        self.sigma = sigma
        self.mean_image: np.ndarray | None = None
        self.size: tuple[int, int] | None = None

    def fit(self, reference_images: Sequence[Image.Image]) -> "MeanImageBaseline":
        self.size = reference_images[0].size
        stack = np.stack(
            [np.asarray(im.resize(self.size), dtype=np.float32) / 255 for im in reference_images]
        )
        self.mean_image = stack.mean(axis=0)
        return self

    def predict(self, image: Image.Image) -> tuple[float, np.ndarray]:
        if self.mean_image is None:
            raise RuntimeError("Call fit() before predict().")
        pixels = np.asarray(image.resize(self.size), dtype=np.float32) / 255
        difference = np.abs(pixels - self.mean_image).mean(axis=2)
        heatmap = gaussian_filter(difference, sigma=self.sigma)
        score = float(np.quantile(heatmap, 0.99))  # robust "maximum"
        return score, heatmap

In [ ]:
def run_experiment(method, subset: str, k: int | None = None, seed: int = SEED):
    # Fit on k reference images (all if k is None), choose the IoU threshold
    # on dev, and report dev and test metrics.
    reference = [s.image for s in load_split(subset, "reference")]
    if k is not None:
        reference = random.Random(seed).sample(reference, k)
    if hasattr(method, "fit"):
        method.fit(reference)

    dev, test = load_split(subset, "dev"), load_split(subset, "test")
    dev_predictions = run_method(method, dev)
    threshold = best_iou_threshold([s.mask for s in dev], [p.heatmap for p in dev_predictions])

    dev_metrics, _ = evaluate(dev, dev_predictions, threshold)
    test_predictions = run_method(method, test)
    test_metrics, test_per_image = evaluate(test, test_predictions, threshold)
    return {"dev": dev_metrics, "test": test_metrics}, test, test_predictions, test_per_image

In [ ]:
baseline_results = {}
for subset in CORE_SUBSETS:
    results, test_samples, test_predictions, per_image = run_experiment(
        MeanImageBaseline(), subset, k=10
    )
    baseline_results[subset] = results["test"]
    print(subset)
    display(by_defect_type(per_image))

pd.DataFrame(baseline_results).T.round(3)

In [ ]:
def show_predictions(
    samples: Sequence[Sample], predictions: Sequence[Prediction], n: int = 4
) -> None:
    # Top row: ground truth outlined. Bottom row: predicted heatmap.
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8), squeeze=False)
    for i, (sample, prediction) in enumerate(list(zip(samples, predictions))[:n]):
        axes[0, i].imshow(outline_mask(sample.image, sample.mask))
        axes[0, i].set_title(sample.defect_type or "normal", fontsize=9)
        axes[1, i].imshow(overlay_heatmap(sample.image, prediction.heatmap))
        axes[1, i].set_title(f"score = {prediction.score:.3f}", fontsize=9)
        for ax in axes[:, i]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


# The last subset from the loop above.
anomalous = [(s, p) for s, p in zip(test_samples, test_predictions) if s.label == 1]
show_predictions(*zip(*anomalous[:4]))

## 6. Zero-shot method: CLIP with text prompts (TODO)

CLIP embeds images and text in the same space, so you can compare an image with prompts such as *"a photo of a flawless cashew"* and *"a photo of a damaged cashew"* without any training.

Your method needs to produce both outputs:

- **Image score:** compare the whole image's embedding with "normal" and "anomalous" prompts.
- **Heatmap:** CLIP gives one embedding per image, so you need a way to get spatial information. Two common options are scoring overlapping crops (sliding windows) and comparing the vision transformer's patch tokens with the text embeddings. Patch tokens aren't trained to align with text, so expect to experiment; WinCLIP (Jeong et al., CVPR 2023) is a good reference for the sliding-window idea.

Things to decide and justify in your report: the prompt wording (and whether to average several prompts), the window or patch scheme, and how you turn similarities into scores.

The model loading is provided below.

In [ ]:
"""
class ZeroShotCLIP:
    # TODO: zero-shot anomaly detection and localization with CLIP.
    def __init__(self, object_name: str) -> None:
        self.object_name = object_name
        self.model = clip_model
        self.processor = clip_processor
        self.model.eval()
        self.text_inputs = self.processor(
            text=[
                f"a photo of a {object_name}",
                f"a photo without a {object_name}",
            ],
            return_tensors="pt",
            padding=True,
        ).DEVICE
        # TODO: define your normal and anomalous prompts, and precompute their
        # text embeddings here so that predict() doesn't recompute them.

    @torch.no_grad()
    def predict(self, image: Image.Image) -> tuple[float, np.ndarray]:
        # TODO: return (score, heatmap). Higher means more anomalous.
        raise NotImplementedError

# When your method is ready:
# results, samples, predictions, per_image = run_experiment(ZeroShotCLIP("cashew"), "cashew")

clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
#https://www.geeksforgeeks.org/deep-learning/zero-shot-learning-for-novel-class-recognition-using-clip-model/
"""

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch16"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)

In [ ]:
class ZeroShotCLIP:
    """Zero-shot anomaly detection and localization using CLIP."""

    def __init__(self, object_name: str) -> None:
        self.object_name = object_name
        self.device = DEVICE

        # Use the provided model loader here instead if one already exists.
        self.model = clip_model.to(self.device)
        self.processor = clip_processor
        self.model.eval()

        # Multiple prompts reduce dependence on a single wording.
        normal_templates = [
            f"a photo of a flawless {object_name}",
            f"a photo of a normal, undamaged {object_name}",
            f"a photo of a good quality {object_name}",
        ]
        anomaly_templates = [
            f"a photo of a damaged {object_name}",
            f"a photo of a defective {object_name}",
            f"a photo of a flawed {object_name}",
        ]

        prompts = normal_templates + anomaly_templates
        tokens = self.processor(
            text=prompts,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
          text_outputs = self.model.text_model(**tokens)
          text_features = self.model.text_projection(
              text_outputs.pooler_output
          )
          text_features = F.normalize(text_features, dim=-1)
        #with torch.no_grad():
         # text_features = clip_model.get_text_features(**tokens)
         # text_features = F.normalize(text_features, dim=-1)

        n = len(normal_templates)

        # Average prompt embeddings within each class, then normalize.
        normal_embedding = F.normalize(
            text_features[:n].mean(dim=0), dim=0
        )
        anomaly_embedding = F.normalize(
            text_features[n:].mean(dim=0), dim=0
        )

        self.text_features = torch.stack(
            [normal_embedding, anomaly_embedding], dim=0
        )

        # Crop and stride are fractions of the shorter image dimension.
        self.window_fraction = 0.5
        self.stride_fraction = 0.25

        # Temperature controls how strongly similarity differences
        # affect the anomaly probability.
        self.temperature = 0.01

    @torch.no_grad()
    def _score_images(self, images: list[Image.Image]) -> np.ndarray:
        """Compute anomaly scores for a batch of images or crops."""
        inputs = self.processor(
            images=images,
            return_tensors="pt",
            padding=True,
        ).to(self.device)

        image_outputs = self.model.vision_model(**inputs)
        image_features = self.model.visual_projection(
                          image_outputs.pooler_output
                            )
        image_features = F.normalize(image_features, dim=-1)

        # Columns: normal similarity, anomalous similarity.
        similarities = image_features @ self.text_features.T

        probabilities = (similarities / self.temperature).softmax(dim=-1)

        # Return the anomalous-class probability.
        return probabilities[:, 1].cpu().numpy()

    @torch.no_grad()
    def predict(self, image: Image.Image) -> tuple[float, np.ndarray]:
        image = image.convert("RGB")
        width, height = image.size

        # 1. Image-level score from the entire image.
        image_score = float(self._score_images([image])[0])

        # 2. Generate overlapping square crops.
        window = max(
            1,
            min(int(min(width, height) * self.window_fraction),
                width, height),
        )
        stride = max(1, int(window * self.stride_fraction))

        def positions(length: int) -> list[int]:
            if length <= window:
                return [0]
            coords = list(range(0, length - window + 1, stride))
            if coords[-1] != length - window:
                coords.append(length - window)
            return coords

        xs = positions(width)
        ys = positions(height)

        crops = []
        boxes = []

        for y in ys:
            for x in xs:
                x2 = min(x + window, width)
                y2 = min(y + window, height)
                crops.append(image.crop((x, y, x2, y2)))
                boxes.append((x, y, x2, y2))

        # Batch crop inference for efficiency.
        crop_scores = []
        batch_size = 32

        for start in range(0, len(crops), batch_size):
            batch = crops[start:start + batch_size]
            crop_scores.extend(self._score_images(batch))

        # 3. Project each crop's score back onto its image region.
        score_sum = np.zeros((height, width), dtype=np.float32)
        count = np.zeros((height, width), dtype=np.float32)

        for score, (x1, y1, x2, y2) in zip(crop_scores, boxes):
            score_sum[y1:y2, x1:x2] += score
            count[y1:y2, x1:x2] += 1.0

        heatmap = score_sum / np.maximum(count, 1.0)

        # Output heatmap has the same H x W dimensions as the input.
        return image_score, heatmap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/CLIP_01"
SAVE_DIR_2 = "/content/drive/MyDrive/SHOTS_01"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR_2, exist_ok=True)

print("Saving files to:", SAVE_DIR, SAVE_DIR_2)


In [ ]:
import torch.nn.functional as F
#results, samples, predictions, per_image = run_experiment(ZeroShotCLIP("cashew"), "cashew")

In [ ]:
"""
import pickle
import os
results_c, samples_c, predictions_c, per_image_c = run_experiment(ZeroShotCLIP("cashew"), "cashew")
with open(os.path.join(SAVE_DIR, "cashew_results.pkl"), "wb") as f:
    pickle.dump(
        {
            "results": results_c,
            "samples": samples_c,
            "predictions": predictions_c,
            "per_image": per_image_c,
        },
        f
    )

print("cashew results saved to Google Drive!")

results_m, samples_m, predictions_m, per_image_m = run_experiment(ZeroShotCLIP("macaroni1"), "macaroni1")
with open(os.path.join(SAVE_DIR, "macaroni1_results.pkl"), "wb") as f:
    pickle.dump(
        {
            "results": results_m,
            "samples": samples_m,
            "predictions": predictions_m,
            "per_image": per_image_m,
        },
        f
    )

print("macaroni1 results saved to Google Drive!")
"""

In [ ]:
import pickle
# Make sure Google Drive is mounted and SAVE_DIR exists
os.makedirs(SAVE_DIR, exist_ok=True)

objects = ["cashew", "macaroni1"]

for obj in objects:
    results, samples, predictions, per_image = run_experiment(
        ZeroShotCLIP(obj), obj
    )

    save_path = os.path.join(SAVE_DIR, f"{obj}_results.pkl")

    with open(save_path, "wb") as f:
        pickle.dump(
            {
                "results": results,
                "samples": samples,
                "predictions": predictions,
                "per_image": per_image,
            },
            f
        )

    print(f"{obj} results saved to {save_path}")

In [ ]:
import pickle
import os

with open(os.path.join(SAVE_DIR, "cashew_results.pkl"), "rb") as f:
    cashew_data = pickle.load(f)

results_c = cashew_data["results"]
samples_c = cashew_data["samples"]
predictions_c = cashew_data["predictions"]
per_image_c = cashew_data["per_image"]

print("Cashew results loaded!")

In [ ]:
with open(os.path.join(SAVE_DIR, "macaroni1_results.pkl"), "rb") as f:
    macaroni_data = pickle.load(f)

results_m = macaroni_data["results"]
samples_m = macaroni_data["samples"]
predictions_m = macaroni_data["predictions"]
per_image_m = macaroni_data["per_image"]

print("Macaroni results loaded!")

## 7. Few-shot method: a patch-feature memory bank (TODO)

The idea, simplified from PatchCore (Roth et al., CVPR 2022):

1. Extract a feature vector for every patch of each of the *k* normal reference images and store them all in a memory bank.
2. For a new image, extract its patch features. Each patch's anomaly score is its distance to the nearest patch in the memory bank. A patch that looks unlike anything in the normal images gets a high score.
3. The patch scores form the heatmap; a high quantile or the maximum gives the image score.

DINOv2 features work well for this, and so do CLIP's vision patch tokens. Decisions to make and justify: the backbone and layer, the input resolution, the distance measure, how you aggregate patch scores into an image score, and any smoothing.

**Required experiment:** report results for *k* = 1, 5, and 20 reference images. How quickly does performance improve with more normal examples, and how does few-shot compare with zero-shot?

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINO_NAME = "facebook/dinov2-small"
dino_model = AutoModel.from_pretrained(DINO_NAME).to(DEVICE).eval()
dino_processor = AutoImageProcessor.from_pretrained(DINO_NAME)

In [ ]:
"""
class FewShotMemoryBank:
    # TODO: few-shot anomaly detection with a memory bank of normal patch features.

    def __init__(self) -> None:
        self.memory_bank: torch.Tensor | None = None  # shape (num_patches_total, feature_dim)

    @torch.no_grad()
    def fit(self, reference_images: Sequence[Image.Image]) -> "FewShotMemoryBank":
        # TODO: extract patch features from each reference image and store them.
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, image: Image.Image) -> tuple[float, np.ndarray]:
        # TODO: return (score, heatmap) using nearest-neighbour distances.
        raise NotImplementedError


# When your method is ready:
# for k in [1, 5, 20]:
#     results, samples, predictions, per_image = run_experiment(FewShotMemoryBank(), "cashew", k=k)
#     print(k, results["test"])
"""

In [ ]:
class FewShotMemoryBank:
    # TODO: few-shot anomaly detection with a memory bank of normal patch features.

    def __init__(self) -> None:
        self.memory_bank: torch.Tensor | None = None  # shape (num_patches_total, feature_dim)
        self.grid_size: tuple[int, int] | None = None

    @torch.no_grad()
    def fit(self, reference_images: Sequence[Image.Image]) -> "FewShotMemoryBank":
        # TODO: extract patch features from each reference image and store them.
        #raise NotImplementedError
        all_features = []

        for image in reference_images:
            inputs = dino_processor(
                images=image.convert("RGB"), return_tensors="pt"
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            outputs = dino_model(**inputs)
            features = F.normalize(outputs.last_hidden_state[:, 1:, :], dim=-1)
            features = features.squeeze(0)

            patch_size = dino_model.config.patch_size
            height, width = inputs["pixel_values"].shape[-2:]
            self.grid_size = (height // patch_size, width // patch_size)

            all_features.append(features)

        self.memory_bank = torch.cat(all_features, dim=0)
        return self

    @torch.no_grad()
    def predict(self, image: Image.Image) -> tuple[float, np.ndarray]:
        # TODO: return (score, heatmap) using nearest-neighbour distances.
        #raise NotImplementedError
        inputs = dino_processor(
        images=image.convert("RGB"), return_tensors="pt"
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        outputs = dino_model(**inputs)
        features = F.normalize(outputs.last_hidden_state[:, 1:, :], dim=-1)
        features = features.squeeze(0)

        # Nearest-neighbour cosine distances
        similarities = features @ self.memory_bank.T
        distances = 1.0 - similarities.max(dim=1).values

        # Reshape patch distances into a heatmap
        grid_h, grid_w = self.grid_size
        patch_map = distances.reshape(1, 1, grid_h, grid_w)

        heatmap = F.interpolate(
            patch_map,
            size=(image.height, image.width),
            mode="bilinear",
            align_corners=False,
        )[0, 0].cpu().numpy()

        score = float(distances.max().item())
        return score, heatmap


# When your method is ready:
# for k in [1, 5, 20]:
#     results, samples, predictions, per_image = run_experiment(FewShotMemoryBank(), "cashew", k=k)
#     print(k, results["test"])

In [ ]:
for k in [1, 5, 20]:
    results, samples, predictions, per_image = run_experiment(FewShotMemoryBank(), "cashew", k=k)
    print(k, results["test"])

## 8. Your experiments (TODO)

Use as many cells as you need. At a minimum, your report should include:

1. **A results table** for the baseline, zero-shot, and few-shot methods on both core subsets: image AUROC, pixel AUROC, mean IoU, and seconds per image on `test`.
2. **The effect of *k*** for the few-shot method (*k* = 1, 5, 20).
3. **A per-defect-type breakdown** (`by_defect_type`) for your best method, with a short interpretation.
4. **A failure analysis** with example images: false positives, missed defects, and wrong localizations. The `defect_area_px` column helps you check whether small defects explain some misses.
5. **A speech robustness check:** record about 10 spoken questions, compare Whisper's transcripts with what you said, and describe what happens downstream when a transcript is wrong (Section 9 has a helper for this).

In [ ]:
# Run the required comparisons. This can take several minutes on a T4 because
# CLIP evaluates overlapping crops. Run this cell once for the final experiment.
# Do not change prompts/parameters after inspecting test results.

import gc

EXPERIMENTS = {}  # (subset, method_name) -> full experiment outputs
METHOD_CONFIGS = {}  # (subset, method_name) -> factory metadata

for subset in CORE_SUBSETS:
    # Classical baseline, using 10 normal reference images.
    method_name = "baseline"
    results, test_samples, test_predictions, per_image = run_experiment(
        MeanImageBaseline(), subset, k=10
    )
    EXPERIMENTS[(subset, method_name)] = {
        "results": results, "samples": test_samples,
        "predictions": test_predictions, "per_image": per_image,
    }
    METHOD_CONFIGS[(subset, method_name)] = {"kind": "baseline", "k": 10}

    # Zero-shot CLIP. No reference images are used.
    method_name = "clip"
    results, test_samples, test_predictions, per_image = run_experiment(
        ZeroShotCLIP(subset), subset
    )
    EXPERIMENTS[(subset, method_name)] = {
        "results": results, "samples": test_samples,
        "predictions": test_predictions, "per_image": per_image,
    }
    METHOD_CONFIGS[(subset, method_name)] = {"kind": "clip", "k": None}
    del test_predictions
    gc.collect()

    # Few-shot results for every required k. Use the same deterministic seed
    # for reproducibility; each run samples only from the normal reference split.
    for k_value in (1, 5, 20):
        method_name = f"fewshot_k{k_value}"
        results, test_samples, test_predictions, per_image = run_experiment(
            FewShotMemoryBank(), subset, k=k_value
        )
        EXPERIMENTS[(subset, method_name)] = {
            "results": results, "samples": test_samples,
            "predictions": test_predictions, "per_image": per_image,
        }
        METHOD_CONFIGS[(subset, method_name)] = {"kind": "fewshot", "k": k_value}
        del test_predictions
        gc.collect()

# Main comparison: baseline, CLIP, and k=5 few-shot on TEST.
main_rows = []
for subset in CORE_SUBSETS:
    for method_name, label in (("baseline", "Baseline (k=10)"),
                               ("clip", "Zero-shot CLIP"),
                               ("fewshot_k5", "Few-shot DINOv2 (k=5)")):
        metrics = EXPERIMENTS[(subset, method_name)]["results"]["test"]
        main_rows.append({"subset": subset, "method": label, **metrics})
main_results_table = pd.DataFrame(main_rows)
display(main_results_table.round(4))

# Effect of k: report each required k on TEST, separately for each subset.
k_rows = []
for subset in CORE_SUBSETS:
    for k_value in (1, 5, 20):
        metrics = EXPERIMENTS[(subset, f"fewshot_k{k_value}")]["results"]["test"]
        k_rows.append({"subset": subset, "k": k_value, **metrics})
k_results_table = pd.DataFrame(k_rows)
display(k_results_table.round(4))

# Select the app's method using DEV metrics only (never test metrics).
# Localization is prioritized, then image discrimination, because the app must
# both flag a defective item and highlight its location.
best_method_by_subset = {}
for subset in CORE_SUBSETS:
    candidates = ["baseline", "clip", "fewshot_k1", "fewshot_k5", "fewshot_k20"]
    best_method_by_subset[subset] = max(
        candidates,
        key=lambda name: (
            EXPERIMENTS[(subset, name)]["results"]["dev"].get("mean_iou", float("-inf")),
            EXPERIMENTS[(subset, name)]["results"]["dev"]["pixel_auroc"],
            EXPERIMENTS[(subset, name)]["results"]["dev"]["image_auroc"],
        ),
    )
print("App methods selected from DEV metrics only:", best_method_by_subset)

# Per-defect-type breakdown for the selected best method on each subset.
for subset in CORE_SUBSETS:
    chosen = best_method_by_subset[subset]
    print(f"\n{subset}: {chosen} — per-defect-type test breakdown")
    per_image = EXPERIMENTS[(subset, chosen)]["per_image"]
    display(by_defect_type(per_image))

# Failure analysis data: image-level false alarms/misses use the app's DEV F1
# threshold; localization failures use the DEV-selected IoU threshold already
# applied by run_experiment. We display examples, not just aggregate counts.
def _dev_f1_threshold(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    best_t, best_f1 = float(scores.min()), -1.0
    for t in np.unique(scores):
        pred = scores >= t
        tp = np.sum(pred & (labels == 1))
        precision = tp / max(int(pred.sum()), 1)
        recall = tp / max(int((labels == 1).sum()), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t

for subset in CORE_SUBSETS:
    chosen = best_method_by_subset[subset]
    exp = EXPERIMENTS[(subset, chosen)]
    test_samples = exp["samples"]
    predictions = exp["predictions"]
    dev_samples = load_split(subset, "dev")
    # Refit a fresh instance to obtain the dev threshold consistently.
    cfg = METHOD_CONFIGS[(subset, chosen)]
    if cfg["kind"] == "baseline":
        reference = [s.image for s in load_split(subset, "reference")]
        app_method = MeanImageBaseline().fit(random.Random(SEED).sample(reference, cfg["k"]))
    elif cfg["kind"] == "clip":
        app_method = ZeroShotCLIP(subset)
    else:
        app_method = FewShotMemoryBank().fit(
            random.Random(SEED).sample([s.image for s in load_split(subset, "reference")], cfg["k"])
        )
    dev_scores = [app_method.predict(s.image)[0] for s in dev_samples]
    app_t = _dev_f1_threshold([s.label for s in dev_samples], dev_scores)
    y_pred = [p.score >= app_t for p in predictions]
    false_positive_idx = [i for i, s in enumerate(test_samples) if s.label == 0 and y_pred[i]]
    false_negative_idx = [i for i, s in enumerate(test_samples) if s.label == 1 and not y_pred[i]]
    print(f"\n{subset} / {chosen}: app threshold={app_t:.4f}; false positives={len(false_positive_idx)}; false negatives={len(false_negative_idx)}")
    for title, indices in (("False alarms", false_positive_idx[:3]), ("Missed defects", false_negative_idx[:3])):
        print(title)
        if indices:
            show_predictions([test_samples[i] for i in indices], [predictions[i] for i in indices], n=len(indices))
        else:
            print("No examples of this error type at the selected DEV threshold.")
    # Lowest-IoU anomalous images are candidates for wrong-localization analysis.
    per_image = exp["per_image"]
    if "iou" in per_image:
        bad = per_image[(per_image["label"] == 1) & per_image["iou"].notna()].nsmallest(3, "iou").index.tolist()
        print("Lowest-IoU anomalous examples (potential localization failures)")
        if bad:
            show_predictions([test_samples[i] for i in bad], [predictions[i] for i in bad], n=len(bad))

# Keep tables available for the report and save a compact machine-readable copy.
RESULTS_DIR = Path("mp3_results")
RESULTS_DIR.mkdir(exist_ok=True)
main_results_table.to_csv(RESULTS_DIR / "main_results_test.csv", index=False)
k_results_table.to_csv(RESULTS_DIR / "fewshot_k_test.csv", index=False)
with open(RESULTS_DIR / "experiment_metrics.json", "w") as f:
    json.dump({f"{subset}/{name}": payload["results"] for (subset, name), payload in EXPERIMENTS.items()}, f, indent=2)
print(f"Saved summary tables to {RESULTS_DIR.resolve()}")


### Main results table
See the experiment outputs generated in Section 8 above. Do not tune parameters using the test results.


### Effect of k
See the experiment outputs generated in Section 8 above. Do not tune parameters using the test results.


### Per-defect-type breakdown
See the experiment outputs generated in Section 8 above. Do not tune parameters using the test results.


### Failure analysis
See the experiment outputs generated in Section 8 above. Do not tune parameters using the test results.


### Speech robustness check
See the experiment outputs generated in Section 8 above. Do not tune parameters using the test results.


## 9. Speech and the app

### Speech to text (provided)

Whisper is loaded through the Hugging Face `pipeline`. `whisper-small` is a reasonable trade-off between accuracy and speed on a T4; try other sizes if you want to measure that trade-off. Whisper detects the spoken language automatically.

In [ ]:
from transformers import pipeline

WHISPER_NAME = "openai/whisper-small"
asr = pipeline("automatic-speech-recognition", model=WHISPER_NAME, device=DEVICE)


def transcribe(audio_path: str) -> str:
    # Transcribe an audio file (any format ffmpeg can read).
    return asr(audio_path)["text"].strip()

### Intent and answers (TODO)

`parse_intent` maps a transcript to an action, and `answer_question` turns the method's output into the text and image the user sees. The versions below are deliberately minimal so that the app runs from the start; replace them with your own.

Keyword matching is a perfectly good approach for intent. If you use a language model instead, explain why in your report.

In [ ]:
INTENTS = ("detect", "locate", "describe")


def parse_intent(question: str) -> str:
    """Map common question phrasings to a supported inspection intent."""
    q = " ".join(str(question).lower().strip().split())
    locate_terms = ("where", "show me", "highlight", "point out", "mark", "locate", "find the defect")
    describe_terms = ("describe", "what do you see", "what is in this image", "tell me about")
    if any(term in q for term in locate_terms):
        return "locate"
    if any(term in q for term in describe_terms):
        return "describe"
    # Detection is the safe default for yes/no questions and unfamiliar wording.
    return "detect"


def answer_question(
    image: Image.Image, question: str, method, threshold: float
) -> tuple[str, Image.Image]:
    intent = parse_intent(question)
    score, heatmap = method.predict(image.convert("RGB"))
    is_defective = score >= threshold
    verdict = "appears defective" if is_defective else "appears normal"

    if intent == "describe":
        answer = (
            f"The inspection score is {score:.3f} (decision threshold {threshold:.3f}); "
            f"the item {verdict}. This app detects anomalies but does not generate a general-purpose caption."
        )
        shown = overlay_heatmap(image, heatmap) if is_defective else image.convert("RGB")
    elif intent == "locate":
        answer = (
            f"The item {verdict} (anomaly score {score:.3f}; threshold {threshold:.3f}). "
            "The overlay indicates regions with higher anomaly scores; it is an estimate, not a confirmed defect mask."
        )
        shown = overlay_heatmap(image, heatmap)
    else:
        answer = f"The item {verdict} (anomaly score {score:.3f}; threshold {threshold:.3f})."
        shown = overlay_heatmap(image, heatmap) if is_defective else image.convert("RGB")
    return answer, shown


### Choosing the decision threshold

The app has to say yes or no, so it needs a threshold on the image score. Choose it on `dev`, for example the threshold that maximizes F1 or the one that keeps false alarms below a rate you pick, and explain your choice. The helper below is a starting point.

In [ ]:
def best_f1_threshold(labels: Sequence[int], scores: Sequence[float]) -> float:
    # Image-score threshold with the highest F1 on the given (dev) data.
    labels, scores = np.asarray(labels), np.asarray(scores)
    best_threshold, best_f1 = float(scores.min()), -1.0
    for t in np.unique(scores):
        predicted = scores >= t
        tp = np.sum(predicted & (labels == 1))
        precision = tp / max(predicted.sum(), 1)
        recall = tp / max((labels == 1).sum(), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        if f1 > best_f1:
            best_threshold, best_f1 = float(t), f1
    return best_threshold

### The Gradio app

The app below uses the baseline so that it works immediately. Once your methods are ready, change `APP_METHODS` to use your best one for each subset.

Allow microphone access when your browser asks. If recording doesn't work in your browser, you can upload an audio file or type the question instead.

In [ ]:
import gradio as gr

APP_SUBSETS = CORE_SUBSETS


def build_app_method(subset: str):
    """Rebuild the method selected from development-set metrics in Section 8."""
    chosen = best_method_by_subset[subset]
    cfg = METHOD_CONFIGS[(subset, chosen)]
    reference = [s.image for s in load_split(subset, "reference")]
    if cfg["kind"] == "baseline":
        method = MeanImageBaseline().fit(random.Random(SEED).sample(reference, cfg["k"]))
    elif cfg["kind"] == "clip":
        method = ZeroShotCLIP(subset)
    else:
        method = FewShotMemoryBank().fit(random.Random(SEED).sample(reference, cfg["k"]))
    return method


def app_threshold(subset: str, method) -> float:
    """Choose the yes/no threshold on DEV only by maximizing F1."""
    dev = load_split(subset, "dev")
    scores = [score for score, _ in (method.predict(s.image) for s in dev)]
    return best_f1_threshold([s.label for s in dev], scores)


APP_METHODS = {subset: build_app_method(subset) for subset in APP_SUBSETS}
APP_THRESHOLDS = {subset: app_threshold(subset, method) for subset, method in APP_METHODS.items()}
print("App method by subset:", best_method_by_subset)
print("App threshold by subset (chosen on dev):", APP_THRESHOLDS)


def handle_request(subset, image, audio_path, typed_question):
    if image is None:
        return "", "Please upload an image first.", None
    if audio_path:
        question = transcribe(audio_path)
    elif typed_question and typed_question.strip():
        question = typed_question.strip()
    else:
        return "", "Please record/upload audio or type a question.", None
    answer, shown = answer_question(
        image.convert("RGB"), question, APP_METHODS[subset], APP_THRESHOLDS[subset]
    )
    return question, answer, shown


with gr.Blocks(title="CS 5330 Visual Inspector") as demo:
    gr.Markdown("## Visual Inspector\nUpload an image, then ask a question about it.")
    with gr.Row():
        with gr.Column():
            subset_input = gr.Dropdown(APP_SUBSETS, value=APP_SUBSETS[0], label="Product")
            image_input = gr.Image(type="pil", label="Image")
            audio_input = gr.Audio(
                sources=["microphone", "upload"], type="filepath", label="Spoken question"
            )
            text_input = gr.Textbox(label="Or type your question")
            ask_button = gr.Button("Ask", variant="primary")
        with gr.Column():
            transcript_output = gr.Textbox(label="What I heard")
            answer_output = gr.Textbox(label="Answer")
            image_output = gr.Image(type="pil", label="Result")

    ask_button.click(
        handle_request,
        inputs=[subset_input, image_input, audio_input, text_input],
        outputs=[transcript_output, answer_output, image_output],
    )

# Set share=True only when you intentionally want a public temporary link.
demo.launch(share=False, debug=False)


### Speech robustness helper

Record your test questions (the app's microphone works, as does your phone), upload them to Colab, and list each file with what you actually said. The helper computes a simple word error rate and checks whether the intent survived transcription.

In [ ]:
def word_error_rate(reference: str, hypothesis: str) -> float:
    # Word-level edit distance divided by the number of reference words.
    def normalize(text: str) -> list[str]:
        return "".join(c.lower() if c.isalnum() or c.isspace() else " " for c in text).split()

    ref, hyp = normalize(reference), normalize(hypothesis)
    distance = np.arange(len(hyp) + 1)
    for i, ref_word in enumerate(ref, start=1):
        previous, distance[0] = distance.copy(), i
        for j, hyp_word in enumerate(hyp, start=1):
            distance[j] = min(
                previous[j] + 1, distance[j - 1] + 1, previous[j - 1] + (ref_word != hyp_word)
            )
    return float(distance[-1] / max(len(ref), 1))


def speech_check(recordings: dict[str, str]) -> pd.DataFrame:
    # recordings maps an audio file path to the sentence you actually said.
    rows = []
    for path, said in recordings.items():
        heard = transcribe(path)
        rows.append(
            {
                "file": Path(path).name,
                "said": said,
                "heard": heard,
                "wer": word_error_rate(said, heard),
                "intent_said": parse_intent(said),
                "intent_heard": parse_intent(heard),
            }
        )
    table = pd.DataFrame(rows)
    table["intent_preserved"] = table["intent_said"] == table["intent_heard"]
    return table


# Record approximately 10 real spoken questions and upload the files to the runtime.
# Replace the paths and reference sentences with your own recordings/transcripts.
# Example:
# recordings = {
#     "q01.m4a": "Is this cashew damaged?",
#     "q02.m4a": "Show me where the crack is.",
# }
# speech_results = speech_check(recordings)
# display(speech_results)
# speech_results.to_csv("mp3_results/speech_robustness.csv", index=False)

## 10. Before you submit

- [ ] Results on `test` for the baseline, zero-shot, and few-shot methods on both core subsets
- [ ] Thresholds and prompts chosen on `dev` only, and your report says how
- [ ] The *k* = 1, 5, 20 experiment
- [ ] A per-defect-type breakdown and a failure analysis with example images
- [ ] The speech robustness check
- [ ] A working app, with your best method plugged in
- [ ] Library versions recorded (Section 1)
- [ ] AI tool use disclosed, and you can explain every design decision in your code
- [ ] The dataset cited (see the dataset README for the BibTeX entry)